<a href="https://colab.research.google.com/github/greek-nlp/benchmark/blob/main/gec_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Greek GEC Benchmark in Colab

This notebook runs the Greek grammatical error correction benchmark from this repository in Google Colab using Ollama inside the Colab runtime.

Notes:
- Free Colab often struggles with 7B or 8B local models.
- Start with one smaller model and a small sample size.
- In Ollama, use `llama3.1:8b`, not `llama3.1:8b-instruct`.


## Repo Setup

Open this notebook from the repository in Colab, or clone/upload the repository files into the Colab runtime before running the benchmark cells.


In [1]:
!apt-get -qq update
!apt-get -qq install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!pip -q install pandas pywer openpyxl

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print("Started Ollama server with PID", ollama_process.pid)

Started Ollama server with PID 3273


In [11]:
# Safer default for Colab memory.
#!ollama pull qwen2.5:3b

# Optional heavier models:
!ollama pull qwen2.5:7b-instruct
!ollama pull aya-expanse:8b
!ollama pull llama3.1:8b

In [18]:
from pathlib import Path
import sys

project_root = Path.cwd()
print("Working directory:", project_root)
print("Python executable:", sys.executable)

from gec_benchmark import benchmark_ollama_models, save_results, load_gec_dataset

MODELS = [
    "qwen2.5:7b-instruct",
    "aya-expanse:8b",
    "llama3.1:8b"
]

SAMPLE_SIZE = 10
RANDOM_STATE = 42
OUTPUT_DIR = "results/gec_ollama_colab"


Working directory: /content
Python executable: /usr/bin/python3


In [19]:
dataset = load_gec_dataset()
print("Dataset size:", len(dataset))
dataset.head()

Dataset size: 175


,original_text,corrected_text
0,...,...
1,...,...
2,...,...
3,...,...
4,...,...


In [20]:
summary, raw = benchmark_ollama_models(
    models=MODELS,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

summary

,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds
2,llama3.1:8b,10,0.3,9.649123,4.260985,10.619469,4.018754,1.0,2.293079
0,qwen2.5:7b-instruct,10,0.2,14.035088,6.790945,12.831858,6.162090,0.9,6.204534
1,aya-expanse:8b,10,0.1,24.122807,14.247670,24.336283,13.998660,1.0,3.750146


In [21]:
save_results(summary, raw, OUTPUT_DIR)
print(f"Saved benchmark outputs to {OUTPUT_DIR}")
summary

Saved benchmark outputs to results/gec_ollama_colab


,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds
2,llama3.1:8b,10,0.3,9.649123,4.260985,10.619469,4.018754,1.0,2.293079
0,qwen2.5:7b-instruct,10,0.2,14.035088,6.790945,12.831858,6.162090,0.9,6.204534
1,aya-expanse:8b,10,0.1,24.122807,14.247670,24.336283,13.998660,1.0,3.750146
